# The `H` node-selector algebra: six worked examples

This notebook is a design prototype, not a released feature. `pygambit.gambit.H` is an
internal, unreleased module -- everything here demonstrates a work-in-progress replacement
for constructing extensive-form games without ever handling raw `Node` objects.

The core idea: a *selector*, built from `H`, describes a set of histories symbolically --
it carries no reference to any particular game until you hand it to one. `H.path(*steps)`
walks a sequence of exact labels and/or `...` wildcards from the root (or from wherever a
selection currently is, when chained); `H.after(*labels)` matches anywhere by a trailing
label pattern; `.plays` expands to whatever is currently terminal; `.by(callable)`
partitions a selection by a key function, and `.filter(callable)` keeps only matching
elements. `Game.append_move`/`append_event`/`append_infoset`/`make_outcome` all accept these
selectors directly, in place of `Node`/`NodeReferenceSet`.

Six examples below, each chosen to exercise a different corner of the design: a classic
imperfect-information game needing `append_infoset`, a game with betting and outcome
computation, a regular two-stage Bayesian game, three different shapes of imperfect
*recall*, and a variation showing `append_infoset` composes normally with further
construction.

In [1]:
try:
    from gtdraw import draw
except ImportError:
    def draw(*args, **kwargs):
        print("gtdraw is not installed; game trees won't be drawn, but everything else runs.")

from pygambit.gambit import H

import pygambit as gbt

## 1. Selten's Horse

A classic three-player game (Selten, 1975) used to illustrate subtleties of sequential
equilibrium. Player 1 moves first; if he plays "R", Player 2 moves; if Player 2 also plays
"L", or if Player 1 played "L" directly, Player 3 faces the same decision either way --
**Player 3 cannot tell which path led there**.

This needs `append_infoset`, not because of anything exotic about recall or timing (the
game is perfectly ordinary on both counts), but for a mundane construction-ordering reason:
Player 3's two infoset members aren't simultaneously available. The node reached via a bare
"L" exists as soon as Player 1 moves; the node reached via "R", "L" only exists once Player 2
has *also* moved -- so one `append_move` call can never cover both.

In [2]:
g = gbt.Game.new_tree(players=["Player 1", "Player 2", "Player 3"], title="Selten's Horse")

g.append_move(H.path(), "Player 1", ["R", "L"])
g.append_move(H.path("L"), "Player 3", ["R", "L"])
g.append_move(H.path("R"), "Player 2", ["R", "L"])
g.append_infoset(H.path("R", "L"), H.path("L"))

g.make_outcome(H.path("R", "R"), {"Player 1": 1, "Player 2": 1, "Player 3": 1}, "RR")
g.make_outcome(H.path("R", "L", "R"), {"Player 1": 4, "Player 2": 4, "Player 3": 0}, "RLR")
g.make_outcome(H.path("R", "L", "L"), {"Player 1": 0, "Player 2": 0, "Player 3": 1}, "RLL")
g.make_outcome(H.path("L", "R"), {"Player 1": 3, "Player 2": 2, "Player 3": 2}, "LR")
g.make_outcome(H.path("L", "L"), {"Player 1": 0, "Player 2": 0, "Player 3": 0}, "LL")

draw(g)
print("is_perfect_recall:", g.is_perfect_recall)
print(
    "Player 3's infoset members (as Histories, not raw Node paths -- the latter"
    " display node-to-root, easy to misread):",
    sorted(g._get_histories(H.path("L")) + g._get_histories(H.path("R", "L"))),
)

is_perfect_recall: True
Player 3's infoset members (as Histories, not raw Node paths -- the latter display node-to-root, easy to misread): [('L',), ('R', 'L')]


## 2. Kuhn poker

Three-card poker, the standard small illustration of imperfect information *and* betting.
This example exercises the recall-tracking machinery in earnest: Alice's second decision
(call/fold after checking then facing a bet) must still distinguish her own card, even
though the tree has grown well past where that distinction was first established.

`alice_partition` is built once, tagged `.with_recall("Alice")`, and reused for both of her
decisions -- the tag makes `.plays` automatically fold her own last action into the group
key from her second decision onward, with no separate re-derivation step. `bob_partition`
never needs the tag: his two uses are his *one* decision instantiated on two mutually
exclusive branches, not a first-then-second sequence for him.

Outcome computation is a genuinely different kind of selector from the recall-tracking
above: `winner`/`pot_size` are direct, declarative facts about a completed hand (who took
the pot, how much), not a player's own partial view of the game -- an outcome deliberately
throws away *how* a given payoff was reached, which is the opposite spirit from recall
grouping's insistence on never conflating what a player can actually tell apart.

In [3]:
CARD_VALUE = {"J": 0, "Q": 1, "K": 2}
cards = list(CARD_VALUE)

g = gbt.Game.new_tree(players=["Alice", "Bob"], title="Kuhn poker")
g.append_event(H.path(), cards, [gbt.Rational(1, 3)] * 3)
for c in cards:
    remaining = [x for x in cards if x != c]
    g.append_event(H.path(c), remaining, [gbt.Rational(1, 2)] * 2)

alice_partition = H.path(...).by(lambda h: h[0]).with_recall("Alice")
g.append_move(alice_partition.plays, "Alice", ["Check", "Bet"])

bob_partition = H.path(..., ...).by(lambda h: h[1])
g.append_move(bob_partition.plays.after("Check"), "Bob", ["Check", "Bet"])

g.append_move(alice_partition.plays.after("Check", "Bet"), "Alice", ["Fold", "Call"])
g.append_move(bob_partition.plays.after("Bet"), "Bob", ["Fold", "Call"])

draw(g)
print("is_perfect_recall:", g.is_perfect_recall)

is_perfect_recall: True


In [4]:
def winner(h):
    match h[2:]:
        case ("Check", "Check") | ("Check", "Bet", "Call") | ("Bet", "Call"):
            return "Alice" if CARD_VALUE[h[0]] > CARD_VALUE[h[1]] else "Bob"
        case ("Check", "Bet", "Fold"):
            return "Bob"
        case ("Bet", "Fold"):
            return "Alice"

def pot_size(h):
    match h[2:]:
        case ("Check", "Check") | ("Check", "Bet", "Fold") | ("Bet", "Fold"):
            return 1
        case ("Check", "Bet", "Call") | ("Bet", "Call"):
            return 2

for (win, amount), group in g._get_groups(H.plays.by(lambda h: (winner(h), pot_size(h)))).items():
    lose = "Bob" if win == "Alice" else "Alice"
    g.make_outcome(group, {win: amount, lose: -amount}, f"{win} wins {amount}")

print("Total outcomes created:", len(list(g.outcomes)))

Total outcomes created: 4


## 3. `bayes2a`: a regular two-stage Bayesian game

A fully "timeable" game with private types and two rounds of simultaneous moves --
`contrib/games/bayes2a.efg` in the repository. Both players privately learn a type, then
move simultaneously each round; each round's actions become public before the next round.

Unlike Kuhn poker, this game needs neither `.with_recall` nor `append_infoset` -- every
player's decision falls at a fixed, predictable position in the history across every
branch, so plain positional indexing on the augmented history object is all the grouping
needs. This is deliberately included as a contrast case: `H`'s dedicated recall machinery
exists for games that need it, but a well-behaved regular game doesn't have to pay for it.

In [5]:
g = gbt.Game.new_tree(players=["Player 1", "Player 2"], title="bayes2a")
half = gbt.Rational(1, 2)

g.append_event(H.path(), ["1G", "1B"], [half, half])
for t1 in ["1G", "1B"]:
    g.append_event(H.path(t1), ["2g", "2b"], [half, half])

# Round 1: each player's move depends only on their own type.
g.append_move(H.path(...).plays.by(lambda h: h[0]), "Player 1", ["H", "L"])
g.append_move(H.path(..., ...).plays.by(lambda h: h[1]), "Player 2", ["h", "l"])

# Round 2: both round-1 actions are now public; each player also still knows their own type.
g.append_move(H.plays.by(lambda h: (h[0], h[2], h[3])), "Player 1", ["H", "L"])
g.append_move(H.plays.by(lambda h: (h[1], h[2], h[3])), "Player 2", ["h", "l"])

PAYOFFS = {
    ("1G", "H", "h"): (10, 2), ("1G", "H", "l"): (0, 10),
    ("1G", "L", "h"): (2, 4),  ("1G", "L", "l"): (4, 0),
    ("1B", "H", "h"): (4, 2),  ("1B", "H", "l"): (2, 10),
    ("1B", "L", "h"): (0, 4),  ("1B", "L", "l"): (10, 0),
}
for (p1, p2), group in g._get_groups(H.plays.by(lambda h: PAYOFFS[(h[0], h[4], h[5])])).items():
    g.make_outcome(group, {"Player 1": p1, "Player 2": p2}, f"({p1},{p2})")

print("is_perfect_recall:", g.is_perfect_recall)
print("terminal histories:", len(g._get_histories(H.plays)))

is_perfect_recall: True
terminal histories: 64


## 4. Imperfect recall: forgetting a past observation

A third, distinct shape of imperfect recall, alongside absent-mindedness and
untimeability below. Alice privately observes a signal (H or L) and acts on it -- her
first decision is correctly split into two infosets, one per signal. Bob then moves,
seeing nothing private. Alice's *second* decision is deliberately built to merge across
both signal values, keyed only by her own first action and Bob's -- she is modeled as
having forgotten the signal that legitimately informed her own first move.

This is neither absent-mindedness (no single node is ever revisited -- these are two
separate first-decision infosets being merged, not one node crossed twice) nor
untimeability (every one of Alice's second-decision nodes sits at exactly the same depth --
the issue is purely about what she remembers, not about timing).

In [6]:
g = gbt.Game.new_tree(players=["Alice", "Bob"], title="Forgetting a past observation")
half = gbt.Rational(1, 2)

g.append_event(H.path(), ["H", "L"], [half, half])
g.append_move(H.path("H"), "Alice", ["Up", "Down"])
g.append_move(H.path("L"), "Alice", ["Up", "Down"])
g.append_move(H.path(..., ...), "Bob", ["x", "y"])

# Keyed by (Alice's own first action, Bob's action) only -- h[0], the signal, is dropped.
g.append_move(H.plays.by(lambda h: (h[1], h[2])), "Alice", ["Fold", "Call"])

draw(g)
print("is_perfect_recall:", g.is_perfect_recall)
# Depth 3, explicitly -- these are the same histories the construction above
# grouped by (h[1], h[2]) to create Alice's second decision.
groups = g._get_groups(H.path(..., ..., ...).by(lambda h: (h[1], h[2])))
for key, members in sorted(groups.items()):
    print(f"  Alice's 2nd decision, key={key}: {sorted(members)}")

is_perfect_recall: False
  Alice's 2nd decision, key=('Down', 'x'): [('H', 'Down', 'x'), ('L', 'Down', 'x')]
  Alice's 2nd decision, key=('Down', 'y'): [('H', 'Down', 'y'), ('L', 'Down', 'y')]
  Alice's 2nd decision, key=('Up', 'x'): [('H', 'Up', 'x'), ('L', 'Up', 'x')]
  Alice's 2nd decision, key=('Up', 'y'): [('H', 'Up', 'y'), ('L', 'Up', 'y')]


## 5. An untimeable game

Jakobsen, Sørensen & Conitzer (2016), Figure 1(a): a coin toss decides who moves first;
each player then guesses whether they went first or second, unable to tell which, since
neither observes the other's move or the coin. Each player's infoset spans both a
depth-1 node (moving first) and depth-2 nodes (moving second) -- and, unlike Selten's
Horse above, **no valid timing assignment exists at all**, even allowing a dense
(non-integer) time scale: each player's second decision would need to come strictly after
the *other's* first decision, on different branches -- a circular constraint no monotonic
timing can resolve. Perfect recall holds throughout regardless -- neither player forgets
anything, each has only one decision to have forgotten at.

In [7]:
g = gbt.Game.new_tree(players=["Player 1", "Player 2"], title="Untimeable (Jakobsen et al. 2016)")

g.append_event(H.path(), ["1", "2"], [gbt.Rational(1, 2)] * 2)

g.append_move(H.path("1"), "Player 2", ["1", "2"])
g.append_move(H.path("2"), "Player 1", ["1", "2"])

g.append_infoset(H.path("1", ...), H.path("2"))
g.append_infoset(H.path("2", ...), H.path("1"))

def outcome_key(h):
    p1_guess = h.last_action("Player 1")
    p2_guess = h.last_action("Player 2")
    return (p1_guess == h[0], p2_guess != h[0])

for (p1_ok, p2_ok), group in g._get_groups(H.plays.by(outcome_key)).items():
    g.make_outcome(
        group, {"Player 1": int(p1_ok), "Player 2": int(p2_ok)},
        f"P1 {'correct' if p1_ok else 'wrong'}, P2 {'correct' if p2_ok else 'wrong'}",
    )

draw(g)
print("is_perfect_recall:", g.is_perfect_recall)

is_perfect_recall: True


## 6. Absent-Minded Driver, with a further decision appended

The classic Piccione–Rubinstein Absent-Minded Driver: one real binary decision ("S"/"T"),
faced *twice* without knowing which time it is, since the driver's own "S"-child shares
her first infoset. This variation goes one step further than the minimal version: after
her second "S", a *second* player gets a genuine, ordinary decision -- showing that
`append_infoset` composes normally with whatever construction comes after it; nothing
about the rest of the tree needs special treatment once the absent-minded infoset is set
up.

In [8]:
g = gbt.Game.new_tree(players=["Player 1", "Player 2"], title="Absent-Minded Driver, extended")

g.append_move(H.path(), "Player 1", ["S", "T"])
g.append_infoset(H.path("S"), H.path())
g.append_move(H.path("S", "T"), "Player 2", ["r", "l"])

g.make_outcome(H.path("S", "S"), {"Player 1": 1, "Player 2": -1}, "SS")
g.make_outcome(H.path("S", "T", "r"), {"Player 1": 2, "Player 2": -2}, "STr")
g.make_outcome(H.path("S", "T", "l"), {"Player 1": 3, "Player 2": -3}, "STl")
g.make_outcome(H.path("T"), {"Player 1": 4, "Player 2": -4}, "T")

draw(g)
print("is_perfect_recall:", g.is_perfect_recall)
print(
    "Player 1's (first) infoset members:",
    sorted(g._get_histories(H.path()) + g._get_histories(H.path("S"))),
)

is_perfect_recall: False
Player 1's (first) infoset members: [(), ('S',)]
